In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()
base_url = "https://hotaruapi.com"
api_key = "sk-d6voa9o4IzfpZJNmhmhsQZTM9RQ9tccDJy4r8OEvd98EFTM8"
client = Anthropic(api_key=api_key, base_url=base_url)
# model = "gemini-3-flash"
model = "claude-sonnet-4-5"


In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    """文本编辑器工具类，提供文件的查看、替换、创建、插入和撤销编辑等操作。
    所有文件操作都限制在指定的基础目录内，防止越权访问。
    修改操作前会自动创建备份，支持撤销。
    """

    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        """初始化文本编辑器工具。

        Args:
            base_dir: 允许操作的基础目录，默认为当前工作目录。
                      所有文件路径都会相对于此目录解析。
            backup_dir: 备份文件存放目录，默认为 base_dir 下的 .backups 子目录。
        """
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        # 确保备份目录存在
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        """验证并解析文件路径，确保路径在允许的基础目录内（防止路径遍历攻击）。

        Args:
            file_path: 相对于 base_dir 的文件路径。

        Returns:
            规范化后的绝对路径。

        Raises:
            ValueError: 如果路径超出允许的目录范围。
        """
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        """在修改文件前创建备份，备份文件名附加修改时间戳以区分不同版本。

        Args:
            file_path: 要备份的文件的绝对路径。

        Returns:
            备份文件的路径；如果源文件不存在则返回空字符串。
        """
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        # 使用文件修改时间作为备份版本标识
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        """从备份目录恢复文件的最新备份版本。

        Args:
            file_path: 要恢复的文件的绝对路径。

        Returns:
            恢复成功的提示信息。

        Raises:
            FileNotFoundError: 如果没有找到该文件的备份。
        """
        file_name = os.path.basename(file_path)
        # 查找该文件的所有备份（按文件名前缀匹配）
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        # 按文件名排序，取最新的备份（时间戳最大的排在最前）
        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        """统计目标字符串在内容中出现的次数，用于替换前校验唯一性。

        Args:
            content: 文件内容。
            old_str: 要查找的字符串。

        Returns:
            匹配次数。
        """
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        """查看文件内容或列出目录内容。输出带有行号前缀，便于定位。

        Args:
            file_path: 要查看的文件或目录路径（相对于 base_dir）。
            view_range: 可选的行范围 [start, end]，1-based 索引。
                        end 为 -1 表示到文件末尾。
                        不指定则显示全部内容。

        Returns:
            带行号的文件内容，或目录下的文件列表。

        Raises:
            FileNotFoundError: 文件不存在。
            PermissionError: 无权限访问。
            UnicodeDecodeError: 文件包含非文本内容。
        """
        try:
            abs_path = self._validate_path(file_path)

            # 如果是目录，则列出目录内容
            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                # 按指定范围截取行（1-based 索引转为 0-based 切片）
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                # 为每行添加行号前缀
                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                # 显示全部内容，带行号
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        """精确替换文件中的一处文本。要求 old_str 在文件中恰好出现一次，
        以避免意外修改。替换前自动创建备份。

        Args:
            file_path: 目标文件路径（相对于 base_dir）。
            old_str: 要被替换的原始文本。
            new_str: 替换后的新文本。

        Returns:
            替换成功的提示信息。

        Raises:
            FileNotFoundError: 文件不存在。
            ValueError: 没有匹配或匹配多于一处（需要提供更多上下文来确保唯一性）。
            PermissionError: 无权限修改文件。
        """
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            # 检查匹配次数，确保恰好匹配一处
            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # 修改前创建备份
            self._backup_file(abs_path)

            # 执行替换
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        """创建新文件并写入内容。如果文件已存在则拒绝创建，
        需使用 str_replace 修改已有文件。会自动创建不存在的父目录。

        Args:
            file_path: 新文件路径（相对于 base_dir）。
            file_text: 要写入的文件内容。

        Returns:
            创建成功的提示信息。

        Raises:
            FileExistsError: 文件已存在。
            ValueError: 路径不合法。
            PermissionError: 无权限创建文件。
        """
        try:
            abs_path = self._validate_path(file_path)

            # 防止覆盖已有文件
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # 自动创建父目录
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # 写入文件
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        """在文件的指定行之后插入新文本。插入前自动创建备份。

        Args:
            file_path: 目标文件路径（相对于 base_dir）。
            insert_line: 在此行号之后插入内容。
                         0 表示插入到文件开头；
                         正数表示插入到对应行之后。
            new_str: 要插入的文本内容。

        Returns:
            插入成功的提示信息。

        Raises:
            FileNotFoundError: 文件不存在。
            IndexError: 行号超出文件范围。
            PermissionError: 无权限修改文件。
        """
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # 修改前创建备份
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # 处理末尾换行：如果文件最后一行没有换行符，先补上换行
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # insert_line == 0 表示插入到文件最开头
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # 在指定行之后插入
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        """撤销对文件的最近一次编辑，从备份中恢复文件到修改前的状态。

        Args:
            file_path: 要撤销编辑的文件路径（相对于 base_dir）。

        Returns:
            恢复成功的提示信息。

        Raises:
            FileNotFoundError: 文件不存在或没有可用的备份。
            PermissionError: 无权限恢复文件。
        """
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

In [4]:
# Process Tool Call Requests
import json

text_editor_tool = TextEditorTool()


def run_tool(tool_name, tool_input):
    print(f"\n 工具名称: {tool_name} \n 工具输入:{tool_input}")
    if tool_name == "str_replace_based_edit_tool":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"],
                tool_input["insert_line"],
                tool_input["new_str"],
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
# 手动定义完整的文本编辑器工具 schema（替代内置的 text_editor_20250728）
# 这样可以兼容代理服务器和非 Anthropic 模型
def get_text_edit_schema(model):
    return {
        "name": "str_replace_based_edit_tool",
        "description": (
            "一个文本编辑器工具，用于查看、创建和编辑文件。\n"
            "支持以下命令：\n"
            "- view: 查看文件内容（带行号）或列出目录内容\n"
            "- create: 创建新文件并写入内容（文件不能已存在）\n"
            "- str_replace: 精确替换文件中的一段文本（必须恰好匹配一处）\n"
            "- insert: 在指定行号之后插入新文本\n"
            "- undo_edit: 撤销对文件的最近一次编辑\n\n"
            "注意事项：\n"
            "- 创建新文件时必须使用 create 命令，不能用 str_replace\n"
            "- str_replace 要求 old_str 在文件中恰好出现一次\n"
            "- 所有路径使用绝对路径"
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "enum": ["view", "create", "str_replace", "insert", "undo_edit"],
                    "description": (
                        "要执行的编辑命令。"
                        "view: 查看文件或目录；"
                        "create: 创建新文件；"
                        "str_replace: 替换文件中的文本；"
                        "insert: 在指定行后插入文本；"
                        "undo_edit: 撤销最近一次编辑"
                    ),
                },
                "path": {
                    "type": "string",
                    "description": "文件或目录的绝对路径。所有命令都需要此参数。",
                },
                "old_str": {
                    "type": "string",
                    "description": (
                        "仅用于 str_replace 命令。要被替换的原始文本，"
                        "必须在文件中恰好出现一次。包含足够的上下文以确保唯一匹配。"
                    ),
                },
                "new_str": {
                    "type": "string",
                    "description": (
                        "用于 str_replace 和 insert 命令。"
                        "str_replace 时为替换后的新文本；"
                        "insert 时为要插入的文本内容。"
                    ),
                },
                "file_text": {
                    "type": "string",
                    "description": "仅用于 create 命令。新文件的完整内容。",
                },
                "view_range": {
                    "type": "array",
                    "items": {"type": "integer"},
                    "description": (
                        "仅用于 view 命令（可选）。"
                        "格式为 [start, end]，1-based 行号索引。"
                        "end 为 -1 表示到文件末尾。不指定则显示全部内容。"
                    ),
                },
                "insert_line": {
                    "type": "integer",
                    "description": (
                        "仅用于 insert 命令。在此行号之后插入内容。"
                        "0 表示插入到文件开头。"
                    ),
                },
            },
            "required": ["command", "path"],
        },
    }

In [6]:
# Run the conversation in a loop until the model doesn't ask for a tool use
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            system=f"你是一个编程助手。当前工作目录为：{os.getcwd()}，所有文件操作都应在此目录下进行。",
            tools=[get_text_edit_schema(model)],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [7]:
messages = []

add_user_message(
    messages,
    """
    0.在当前目录新增一个文件夹，文件夹名称为：test_text_editor
    1. 在test_text_editor中，创建一个main.py文件，并写入以下内容：
    print("Hello, World!")
    2. 在test_text_editor中，在main.py文件中，添加一个函数，函数名称为：
    def add(a, b):
        return a + b
    3. 在test_text_editor中，创建一个带有适当单元测试的新 test.py 文件
    """,
)

run_conversation(messages)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CY9h6fzwCvhJBHCVHLCEz'}

In [ ]:
add_user_message(
    messages,
    """
    在main.py中，添加一个函数，函数名称为：
    def subtract(a, b):
        return a - b
    """,
)

run_conversation(messages)

好的，我来在 main.py 中添加 subtract 函数：

 工具名称: str_replace_based_edit_tool 
 工具输入:{'old_str': 'print("Hello, World!")\n\ndef add(a, b):\n    return a + b', 'new_str': 'print("Hello, World!")\n\ndef add(a, b):\n    return a + b\n\ndef subtract(a, b):\n    return a - b', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api/test_text_editor/main.py', 'command': 'str_replace'}
完成！✅ 

我已经成功在 `main.py` 中添加了 `subtract` 函数。现在 `main.py` 的完整内容为：

```python
print("Hello, World!")

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b
```

函数已添加成功！如果需要，我也可以为这个新函数添加相应的单元测试到 `test.py` 文件中。


[{'role': 'user',
  'content': '\n    0.在当前目录新增一个文件夹，文件夹名称为：test_text_editor\n    1. 在test_text_editor中，创建一个main.py文件，并写入以下内容：\n    print("Hello, World!")\n    2. 在test_text_editor中，在main.py文件中，添加一个函数，函数名称为：\n    def add(a, b):\n        return a + b\n    3. 在test_text_editor中，创建一个带有适当单元测试的新 test.py 文件\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='我来帮你完成这些任务。让我逐步执行：\n\n首先，让我查看当前目录结构，然后创建文件夹和文件。', type='text'),
   ToolUseBlock(id='toolu_vrtx_01TprRnXtbWvY737Rbdr1vqP', input={'command': 'view', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_vrtx_01TprRnXtbWvY737Rbdr1vqP',
    'content': '"tool_use\\n\\u5de5\\u5177\\u4f7f\\u7528(\\u4e00).md\\nrequirements.txt\\n\\u5de5\\u5177\\u4f7f\\u7528(\\u4e00)\\u4e2d\\u6587.md\\n002_tools_batch_and_struct.ipynb\\n003_tool_streaming.ipynb\\n.claude\\n0

In [ ]:
messages = []
add_user_message(
    messages,
    """
    在main.py中，subtract函数前，add函数后，加入一个新的函数，函数名称为：
    def multiply(a, b):
        return a * b
    """,
)

run_conversation(messages)

我来帮你在 main.py 文件中添加这个函数。首先让我查看一下当前文件的内容。

 工具名称: str_replace_based_edit_tool 
 工具输入:{'command': 'view', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api'}
我看到这是一个目录。让我查找 main.py 文件：

 工具名称: str_replace_based_edit_tool 
 工具输入:{'command': 'view', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api/main.py'}
我发现在当前目录下没有找到 main.py 文件。让我检查一下是否在某个子目录中：

 工具名称: str_replace_based_edit_tool 
 工具输入:{'command': 'view', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api/test_text_editor'}
找到了！main.py 在 test_text_editor 目录中。让我查看它的内容：

 工具名称: str_replace_based_edit_tool 
 工具输入:{'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api/test_text_editor/main.py', 'command': 'view'}
好的！我看到了 main.py 的内容。现在我将在 add 函数后、subtract 函数前添加 multiply 函数：

 工具名称: str_replace_based_edit_tool 
 工具输入:{'command': 'str_replace', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api/test_text_edi

[{'role': 'user',
  'content': '\n    在main.py中，subtract函数前，add函数后，加入一个新的函数，函数名称为：\n    def multiply(a, b):\n        return a * b\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='我来帮你在 main.py 文件中添加这个函数。首先让我查看一下当前文件的内容。', type='text'),
   ToolUseBlock(id='toolu_vrtx_01CF83vVPohZ3NKj1cyGnym9', input={'command': 'view', 'path': '/Users/canglong/Documents/deep-learning/building_with_claudeCode_api'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_vrtx_01CF83vVPohZ3NKj1cyGnym9',
    'content': '"tool_use\\n\\u5de5\\u5177\\u4f7f\\u7528(\\u4e00).md\\nrequirements.txt\\n\\u5de5\\u5177\\u4f7f\\u7528(\\u4e00)\\u4e2d\\u6587.md\\n002_tools_batch_and_struct.ipynb\\n003_tool_streaming.ipynb\\n.claude\\n006_web_search_complete.ipynb\\nimage\\nREADME.md\\ntest_text_editor\\n003_tool_streaming_completed.ipynb\\n.gitignore\\nprompt\\n001_tools_1.ipynb\\n.ipynb_checkpoints\\n.backups\\